# Lesson 6 — schema migration · เปลี่ยน schema แล้วย้อนกลับได้

เพิ่ม column · เปลี่ยนชื่อ · ลบ column แต่ละอย่างคือ version ใหม่
ทำพลาดก็ `restore` กลับไป version ก่อนหน้า ไม่ต้องมี migration file ย้อนกลับ
บทนี้ทำทั้งชุด แล้วดู version ทีละขั้น

In [1]:
%pip install -q lancedb pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import lancedb

db = lancedb.connect("./data")
tbl = db.create_table("users", data=[
    {"id": 1, "name": "nat",  "plan": "team"},
    {"id": 2, "name": "beta", "plan": "pro"},
], mode="overwrite")

def show(label):
    print(f"--- v{tbl.version} {label}")
    print(tbl.to_pandas().to_string(index=False))

show("start")

--- v1 start
 id name plan
  1  nat team
  2 beta  pro


[2026-09-10T06:34:48Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/06-migration/data/users.lance, it will be created


**Add column** ค่าเริ่มต้นเขียนเป็น SQL expression
ใส่ค่าคงที่ก็ได้ คำนวณจาก column เดิมก็ได้

In [3]:
tbl.add_columns({"credits": "0", "name_upper": "upper(name)"})
show("add_columns")

--- v2 add_columns
 id name plan  credits name_upper
  1  nat team        0        NAT
  2 beta  pro        0       BETA


**Rename** ผ่าน `alter_columns` ระบุ `path` เดิม กับ `rename` ใหม่
ไฟล์ data ไม่ถูกเขียนใหม่ เปลี่ยนแค่ใน manifest

In [4]:
tbl.alter_columns({"path": "plan", "rename": "tier"})
show("rename plan -> tier")

--- v3 rename plan -> tier
 id name tier  credits name_upper
  1  nat team        0        NAT
  2 beta  pro        0       BETA


**Change type** `alter_columns` รับ `data_type` ก็จริง
แต่ cast ข้ามตระกูล int → float Lance 0.38 ปฏิเสธ
ลองดูก่อน อ่าน error ให้ครบ

In [5]:
import pyarrow as pa
try:
    tbl.alter_columns({"path": "credits", "data_type": pa.float64()})
except ValueError as e:
    print("refused:", e)

refused: Invalid input, Cannot cast column "credits" from Int64 to Float64


วิธีที่ใช้ได้จริง สามขั้น
add column ใหม่ที่ cast มาจากของเก่า → drop ของเก่า → rename ใหม่ให้ชื่อเดิม
ได้ 3 version แต่ทุกขั้นย้อนได้

In [6]:
tbl.add_columns({"credits_f": "CAST(credits AS DOUBLE)"})
tbl.drop_columns(["credits"])
tbl.alter_columns({"path": "credits_f", "rename": "credits"})
print(tbl.schema.field("credits"))
show("credits int64 -> float64")

pyarrow.Field<credits: double not null>
--- v6 credits int64 -> float64
 id name tier name_upper  credits
  1  nat team        NAT      0.0
  2 beta  pro       BETA      0.0


**Drop column** ลบออกจาก schema
ข้อมูลใน fragment เดิมยังอยู่ แค่ manifest ไม่ชี้ไปหาอีก

In [7]:
tbl.drop_columns(["name_upper"])
show("drop name_upper")

--- v7 drop name_upper
 id name tier  credits
  1  nat team      0.0
  2 beta  pro      0.0


**ดูประวัติ** ทุก version มี timestamp
manifest แต่ละอันคือ snapshot ของ schema + fragment ที่ใช้อยู่ตอนนั้น

In [8]:
for v in tbl.list_versions():
    m = v["metadata"]
    print(f"v{v['version']}  data_files={m['total_data_files']}  bytes={m['total_files_size']}")

v1  data_files=1  bytes=925
v2  data_files=2  bytes=1569
v3  data_files=2  bytes=1569
v4  data_files=3  bytes=1918
v5  data_files=3  bytes=1918
v6  data_files=3  bytes=1918
v7  data_files=2  bytes=1274


**ย้อนดู** `checkout(1)` เปิด version 1 แบบอ่านอย่างเดียว
เห็น schema แรกสุด `plan` ยังอยู่ `credits` ยังไม่มี

In [9]:
tbl.checkout(1)
show("checkout 1 (read-only)")

--- v1 checkout 1 (read-only)
 id name plan
  1  nat team
  2 beta  pro


**Restore** ทำให้ version ที่ checkout อยู่ กลายเป็น version ล่าสุด
ไม่ได้ลบ version กลางทาง แค่สร้าง version ใหม่ที่หน้าตาเหมือน version 1
Nothing is Deleted

In [10]:
tbl.restore()
show("after restore")
print("versions kept:", [v["version"] for v in tbl.list_versions()])

--- v8 after restore
 id name plan
  1  nat team
  2 beta  pro
versions kept: [1, 2, 3, 4, 5, 6, 7, 8]
